In [5]:
import duckdb
import pandas as pd

# Connect to database
conn = duckdb.connect('indian_banking.duckdb')

# Helper function to run queries
def query(sql):
    return conn.execute(sql).fetchdf()

print("✓ Connected to database!")
print(f"Total customers: {query('SELECT COUNT(*) as count FROM customers')['count'][0]:,}")

✓ Connected to database!
Total customers: 100,000


### Customer Demographics

In [ ]:
# Count of customers by gender and account type
query("""SELECT gender, account_type, COUNT(*) AS total_customers
FROM customers
GROUP BY gender, account_type
ORDER BY gender, total_customers DESC""")

,gender,account_type,total_customers
0,Female,Savings,25534
1,Female,Current,13438
2,Female,Salary,11022
3,Male,Savings,25522
4,Male,Current,13371
5,Male,Salary,11113


In [18]:
# Average age per account type
query("""SELECT 
    account_type, 
    ROUND(AVG(DATEDIFF('day', date_of_birth, CURRENT_DATE)/365.0), 1) AS avg_age
FROM customers
WHERE date_of_birth IS NOT NULL
GROUP BY account_type""")

,account_type,avg_age
0,Current,46.3
1,Savings,45.3
2,Salary,46.4


In [19]:
# Top 5 occupations by average income
query("""
SELECT occupation, AVG(annual_income) AS avg_income
FROM customers
GROUP BY occupation
ORDER BY avg_income DESC
LIMIT 5""")

,occupation,avg_income
0,Business Owner,5.187198e+06
1,Entrepreneur,4.033795e+06
2,Pilot,3.225061e+06
3,Doctor,2.924540e+06
4,Lawyer,1.766554e+06


So the gender split is almost 50-50 which is interesting. Savings accounts are clearly the most popular — about 51K customers have them. Age-wise, all account types hover around 45-46 years, so the customer base is mostly middle-aged. Business owners and entrepreneurs are the top earners by a big margin (52L and 40L avg), which makes sense.

### Account & Balance Analysis

In [20]:
# Customer segmentation by balance & credit score
query("""
SELECT 
CASE
  WHEN account_balance < 50000 THEN 'Low Value'
  WHEN account_balance BETWEEN 50000 AND 200000 THEN 'Medium Value'
  ELSE 'High Value'
END AS balance_segment,
CASE
  WHEN credit_score >= 750 THEN 'Excellent'
  WHEN credit_score >= 650 THEN 'Good'
  WHEN credit_score >= 550 THEN 'Average'
  ELSE 'Poor'
END AS credit_segment,
COUNT(*) AS total_customers
FROM customers
GROUP BY balance_segment, credit_segment""")

,balance_segment,credit_segment,total_customers
0,Low Value,Good,463
1,High Value,Poor,34946
2,Low Value,Average,477
3,High Value,Good,14157
4,Medium Value,Good,2141
5,High Value,Average,14046
6,Medium Value,Excellent,3204
7,Low Value,Poor,1173
8,Medium Value,Poor,5403
9,Low Value,Excellent,734


In [21]:
# Average balance per bank
query("""
SELECT bank_name, AVG(account_balance) AS avg_balance
FROM customers
GROUP BY bank_name
ORDER BY avg_balance DESC""")

,bank_name,avg_balance
0,Central Bank of India,1.309986e+06
1,State Bank of India,1.304366e+06


In [31]:
# Top 10 customers with highest balance per bank
query("""
SELECT c1.bank_name, c1.full_name, c1.account_balance
FROM customers c1
WHERE c1.account_balance = (
    SELECT MAX(c2.account_balance)
    FROM customers c2
    WHERE c2.bank_name = c1.bank_name
)
""")

,bank_name,full_name,account_balance
0,State Bank of India,Myra Yadav,4999708.28
1,Central Bank of India,Neha Rani Gupta,4999922.19


In [32]:
# Percentage of high-value customers per bank
query("""SELECT bank_name,
COUNT(*) AS total_customers,
SUM(CASE WHEN account_balance > 200000 THEN 1 ELSE 0 END)/COUNT(*)*100 AS high_value_pct
FROM customers
GROUP BY bank_name""")

,bank_name,total_customers,high_value_pct
0,State Bank of India,49979,84.257388
1,Central Bank of India,50021,84.214630


84% of customers have balances above 2 lakhs — that's a lot of high-value customers. Both banks are almost identical here. The worrying part is the biggest segment is high-balance customers with poor credit (about 35K people). That's a risk worth looking into. Average balance per bank is around 13 lakhs.

### Transaction Behavior

In [33]:
# Top 10 most active customers (transactions per month)
query("""SELECT full_name, bank_name, transactions_per_month
FROM customers
ORDER BY transactions_per_month DESC
LIMIT 10""")

,full_name,bank_name,transactions_per_month
0,Myra Prasad Kapoor,State Bank of India,150
1,Vivaan Patel,Central Bank of India,150
2,Priya Saxena,Central Bank of India,150
3,Ira Rani Agarwal,State Bank of India,150
4,Geeta Kumar Das,State Bank of India,150
5,Navya Sen,State Bank of India,150
6,Harsh Mehta,State Bank of India,150
7,Suresh Mohan Yadav,State Bank of India,150
8,Kiara Anand Ghosh,Central Bank of India,150
9,Vivaan Pillai,Central Bank of India,150


In [34]:
# Average monthly transactions by account type
query("""SELECT account_type, AVG(transactions_per_month) AS avg_transactions
FROM customers
GROUP BY account_type""")

,account_type,avg_transactions
0,Current,77.417509
1,Savings,77.707086
2,Salary,77.416806


In [44]:
# Inactive accounts in the last year
query("""
SELECT full_name, bank_name, last_transaction_date
FROM customers
WHERE last_transaction_date < CURRENT_DATE - INTERVAL '1 year'
""")

,full_name,bank_name,last_transaction_date
0,Rohit Bhatia,Central Bank of India,2024-10-24
1,Vihaan Shah,State Bank of India,2024-04-20
2,Karan Shankar Bhatia,Central Bank of India,2024-03-25
3,Sunita Lal Menon,State Bank of India,2024-10-21
4,Ravi Rani Bansal,Central Bank of India,2024-06-18
...,...,...,...
14843,Pallavi Lakshmi Shah,Central Bank of India,2024-08-19
14844,Meera Kumar Mukherjee,Central Bank of India,2024-03-30
14845,Anvi Bhai Yadav,State Bank of India,2024-03-17
14846,Sneha Bansal,Central Bank of India,2024-06-21


In [45]:
# Correlation between balance and transactions per month
query("""
SELECT account_balance, transactions_per_month
FROM customers
ORDER BY account_balance DESC""")

,account_balance,transactions_per_month
0,4999922.19,83
1,4999912.51,114
2,4999708.28,84
3,4999706.44,31
4,4999488.26,82
...,...,...
99995,5074.08,23
99996,5069.95,94
99997,5059.99,96
99998,5046.14,74


Transaction patterns are pretty uniform — around 77 per month regardless of account type. About 15K accounts haven't had any activity in over a year, which is roughly 15% of the total. These could be potential churn cases.

### Credit Score Insights

In [46]:
# Credit score distribution
query("""
SELECT 
CASE
  WHEN credit_score >= 750 THEN 'Excellent'
  WHEN credit_score >= 650 THEN 'Good'
  WHEN credit_score >= 550 THEN 'Average'
  ELSE 'Poor'
END AS credit_category,
COUNT(*) AS total_customers
FROM customers
GROUP BY credit_category""")

,credit_category,total_customers
0,Average,16692
1,Good,16761
2,Poor,41522
3,Excellent,25025


In [47]:
# Average balance per credit score category
query("""SELECT 
CASE
  WHEN credit_score >= 750 THEN 'Excellent'
  WHEN credit_score >= 650 THEN 'Good'
  WHEN credit_score >= 550 THEN 'Average'
  ELSE 'Poor'
END AS credit_category,
AVG(account_balance) AS avg_balance
FROM customers
GROUP BY credit_category""")

,credit_category,avg_balance
0,Average,1.306858e+06
1,Poor,1.304121e+06
2,Excellent,1.316193e+06
3,Good,1.301607e+06


In [48]:
# Banks with highest number of poor credit customers
query("""
SELECT bank_name, COUNT(*) AS poor_credit_count
FROM customers
WHERE credit_score < 550
GROUP BY bank_name
ORDER BY poor_credit_count DESC""")

,bank_name,poor_credit_count
0,Central Bank of India,20804
1,State Bank of India,20718


This one's a bit alarming — 41.5% of customers have credit scores below 550. That's the single largest credit group. Only about 25K have excellent scores (750+). Both banks have similar numbers here, so it's not a bank-specific issue.

### Income & Occupation Analysis

In [49]:
# Average income by gender
query("""SELECT gender, AVG(annual_income) AS avg_income
FROM customers
GROUP BY gender""")

,gender,avg_income
0,Female,1.069039e+06
1,Male,1.061495e+06


In [50]:
# Top 5 banks with highest average customer income
query("""SELECT bank_name, AVG(annual_income) AS avg_income
FROM customers
GROUP BY bank_name
ORDER BY avg_income DESC
LIMIT 5""")

,bank_name,avg_income
0,Central Bank of India,1.068580e+06
1,State Bank of India,1.061951e+06


In [51]:
# Most common occupation per bank
query("""SELECT bank_name, occupation, COUNT(*) AS total_customers
FROM customers
GROUP BY bank_name, occupation
ORDER BY bank_name, total_customers DESC""")

,bank_name,occupation,total_customers
0,Central Bank of India,Pilot,1236
1,Central Bank of India,Real Estate Agent,1183
2,Central Bank of India,Accountant,1177
3,Central Bank of India,Driver,1170
4,Central Bank of India,Content Writer,1166
...,...,...,...
83,State Bank of India,Photographer,1093
84,State Bank of India,Teacher,1089
85,State Bank of India,Lawyer,1076
86,State Bank of India,Architect,1068


Income is pretty evenly distributed between genders. Both banks also attract similar income profiles. There's good occupation diversity across both banks.

### Account Status & Risk

In [52]:
# Active vs dormant accounts per bank
query("""SELECT bank_name,
SUM(CASE WHEN account_status='Active' THEN 1 ELSE 0 END) AS active_accounts,
SUM(CASE WHEN account_status='Inactive' THEN 1 ELSE 0 END) AS dormant_accounts
FROM customers
GROUP BY bank_name""")

,bank_name,active_accounts,dormant_accounts
0,State Bank of India,37361.0,12618.0
1,Central Bank of India,37477.0,12544.0


In [57]:
# Potential churn customers (no transaction in 6+ months)
query("""SELECT full_name, bank_name, last_transaction_date
FROM customers
WHERE last_transaction_date < CURRENT_DATE - INTERVAL '6 months' """)

,full_name,bank_name,last_transaction_date
0,Rohit Bhatia,Central Bank of India,2024-10-24
1,Pooja Sen,Central Bank of India,2025-08-25
2,Vihaan Shah,State Bank of India,2024-04-20
3,Karan Shankar Bhatia,Central Bank of India,2024-03-25
4,Sunita Lal Menon,State Bank of India,2024-10-21
...,...,...,...
22059,Anvi Bhai Yadav,State Bank of India,2024-03-17
22060,Sneha Bansal,Central Bank of India,2024-06-21
22061,Atharv Devi Dubey,State Bank of India,2024-07-11
22062,Rohan Prasad Tiwari,Central Bank of India,2025-07-02


In [58]:
# Average credit score of dormant vs active accounts
query("""SELECT account_status, AVG(credit_score) AS avg_credit
FROM customers
GROUP BY account_status""")

,account_status,avg_credit
0,Inactive,600.194221
1,Active,599.727999


Active accounts outnumber dormant ones in both banks, which is good. But there are quite a few customers who haven't transacted in 6+ months — these are churn risks. Dormant accounts tend to have slightly lower credit scores too.

### Bank Level Insights

In [59]:
# Total deposits per bank
query("""
SELECT bank_name, SUM(account_balance) AS total_deposits
FROM customers
GROUP BY bank_name
ORDER BY total_deposits DESC""")

,bank_name,total_deposits
0,Central Bank of India,6.552683e+10
1,State Bank of India,6.519091e+10


In [60]:
# Average income of top 3 banks by deposits
query("""SELECT bank_name, AVG(annual_income) AS avg_income
FROM customers
WHERE bank_name IN (
  SELECT bank_name
  FROM customers
  GROUP BY bank_name
  ORDER BY SUM(account_balance) DESC
  LIMIT 3
)
GROUP BY bank_name""")

,bank_name,avg_income
0,Central Bank of India,1.068580e+06
1,State Bank of India,1.061951e+06


In [63]:
# Customer growth trend by creation date (monthly)
query("""SELECT strftime('%Y-%m', created_date) AS month, COUNT(*) AS new_customers
FROM customers
GROUP BY month
ORDER BY month""")

,month,new_customers
0,2016-03,725
1,2016-04,841
2,2016-05,823
3,2016-06,803
4,2016-07,857
...,...,...
116,2025-11,787
117,2025-12,886
118,2026-01,802
119,2026-02,716


In [64]:
# Average transactions per month by bank
query("""SELECT bank_name, AVG(transactions_per_month) AS avg_transactions
FROM customers
GROUP BY bank_name""")

,bank_name,avg_transactions
0,State Bank of India,77.565938
1,Central Bank of India,77.564463


In [65]:
# Bank risk analysis (high credit risk customers with high balance)
query("""SELECT bank_name, COUNT(*) AS risky_customers
FROM customers
WHERE credit_score < 550 AND account_balance > 100000
GROUP BY bank_name
ORDER BY risky_customers DESC""")

,bank_name,risky_customers
0,Central Bank of India,19343
1,State Bank of India,19306


In [ ]:
# 

Both SBI and Central Bank show very similar numbers across deposits, income, and transactions.
Customer growth trends could help spot seasonal patterns in new account creation.
The high-balance + poor-credit combo is probably the biggest risk finding here.

---

## Wrapping Up

After going through 100K customer records from SBI and Central Bank of India, here's what stands out:

- 84% customers are high-value (balance > 2L) — strong deposit base for both banks
- But 41.5% have poor credit scores, and a lot of them are the same high-balance customers — that's risky
- Around 15K accounts are inactive for over a year
- Business owners earn the most (~52L/year avg)
- Both banks have almost identical customer profiles across every metric

If I were to suggest next steps:
- Focus on improving credit scores for those 41K+ poor-credit customers
- Try to re-engage the 15K inactive accounts before they churn
- Keep an eye on the high-balance + low-credit segment — that's where the risk is concentrated
- Since both banks look so similar, there might be room for differentiated products